# Analysis Report

Contains exploratory results, revenue analysis, and agronomic insights derived from the cleaned master data.

In [ ]:
# Agronomic analysis
print("\n" + "=" * 60)
print("AGRONOMIC ANALYSIS")
print("=" * 60)

print("\nFertilizer bands:")
fert_bands = [0, 20, 50, 100, float('inf')]
fert_labels = ['0-20', '20-50', '50-100', '100+']
train['fert_band'] = pd.cut(train['fertilizer_kg_per_ha'], bins=fert_bands, labels=fert_labels)
print(train.groupby('fert_band', observed=True)['yield_tons_per_ha'].agg(['mean', 'count']))

print("\nAltitude bands:")
alt_bands = [0, 1500, 2000, 2500, float('inf')]
alt_labels = ['<1500m', '1500-2000m', '2000-2500m', '>2500m']
train['alt_band'] = pd.cut(train['altitude_m'], bins=alt_bands, labels=alt_labels)
print(train.groupby('alt_band', observed=True)['yield_tons_per_ha'].agg(['mean', 'count']))

print("\nPlanting month effect:")
print(train.groupby('planting_month')['yield_tons_per_ha'].agg(['mean', 'count']))

In [ ]:
# Categorical analysis
print("\n" + "=" * 60)
print("CATEGORICAL ANALYSIS")
print("=" * 60)

print("\nBy Region:")
for region in sorted(train['region'].unique()):
    data = train[train['region'] == region]['yield_tons_per_ha']
    print(f"  {region}: mean={data.mean():.3f}, median={data.median():.3f}, std={data.std():.3f}")

print("\nBy Crop:")
for crop in sorted(train['crop_type'].unique()):
    data = train[train['crop_type'] == crop]['yield_tons_per_ha']
    print(f"  {crop}: mean={data.mean():.3f}, median={data.median():.3f}, std={data.std():.3f}")

print("\nImproved seed effect:")
no_seed = train[train['improved_seed_used'] == 0]['yield_tons_per_ha'].mean()
with_seed = train[train['improved_seed_used'] == 1]['yield_tons_per_ha'].mean()
print(f"  Without improved seed: {no_seed:.3f} t/ha")
print(f"  With improved seed: {with_seed:.3f} t/ha")
print(f"  Improvement: {((with_seed/no_seed - 1) * 100):.1f}%")

print("\nPest/Disease effect:")
no_pest = train[train['pest_disease_flag'] == 0]['yield_tons_per_ha'].mean()
with_pest = train[train['pest_disease_flag'] == 1]['yield_tons_per_ha'].mean()
print(f"  No pest/disease: {no_pest:.3f} t/ha")
print(f"  With pest/disease: {with_pest:.3f} t/ha")
print(f"  Impact: {((with_pest/no_pest - 1) * 100):.1f}%")

In [ ]:
# Revenue analysis
print("=" * 60)
print("REVENUE & PRICE ANALYSIS")
print("=" * 60)

revenue = train[['plot_id', 'crop_type', 'region', 'survey_year', 'yield_tons_per_ha']].merge(
    price[['crop_type', 'region', 'year', 'price_birr_per_quintal']],
    left_on=['crop_type', 'region', 'survey_year'],
    right_on=['crop_type', 'region', 'year'],
    how='left'
)

revenue['revenue_per_ha'] = revenue['yield_tons_per_ha'] * 10 * revenue['price_birr_per_quintal']

print(f"\nMean yield by crop:")
print(train.groupby('crop_type')['yield_tons_per_ha'].mean().sort_values(ascending=False))

print(f"\nMean revenue by crop (where price available):")
print(revenue.groupby('crop_type')['revenue_per_ha'].mean().sort_values(ascending=False))

print(f"\nTop 5 revenue-generating crop-region combos:")
combo_revenue = revenue.groupby(['crop_type', 'region'])['revenue_per_ha'].mean().sort_values(ascending=False)
print(combo_revenue.head())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')

# Load data
train = pd.read_csv(PROCESSED_DIR / 'master_train.csv')
price = pd.read_csv(RAW_DIR / 'market_prices.csv')

print("Master train shape:", train.shape)
print("\nColumns:", list(train.columns)[:15], "...")
print("\nFirst few rows:")
print(train.head())